# 03 · Rich Compact Student v2 — No KD

This replaces the ultra-compressed 670k v0 student as the active architecture candidate.

### Verified design

**Audio input**
- reuses the already extracted patient-only USSD-compatible ComParE16 files
- shape before segmentation: `130 × variable_frames`
- 384 frames per acoustic segment
- up to 32 uniformly distributed segments at DEV
- TRAIN uses deterministic epoch-varying segment sampling
- normalization is fitted again from TRAIN-107 only

**Text input**
- full Participant-only interview document
- independent TRAIN-only TF-IDF
- top 250 selected with `SelectKBest(f_classif)`
- the frozen Idiap vectorizer is **not** used

**Student**
- temporal Conv1D + depthwise temporal Conv + small GRU audio encoder
- compact text MLP
- text-conditioned attention across acoustic segments
- participant-level classifier
- about 260k trainable parameters

No teacher probabilities, logits, hidden states, or checkpoints enter this baseline. TEST remains closed.


In [ ]:
from google.colab import drive,userdata
drive.mount('/content/drive')

from pathlib import Path
import base64,subprocess,sys,json,shutil
REPO='engrkhubabahmad/reliability-aware-depression-kd'
ROOT=Path('/content/reliability-aware-depression-kd')
token=userdata.get('GITHUB_TOKEN'); assert token,'Add GITHUB_TOKEN to Colab Secrets'
auth=base64.b64encode(f'x-access-token:{token}'.encode()).decode()
git=['git','-c',f'http.extraHeader=AUTHORIZATION: basic {auth}']
if not (ROOT/'.git').exists(): subprocess.run(git+['clone',f'https://github.com/{REPO}.git',str(ROOT)],check=True)
subprocess.run(git+['-C',str(ROOT),'fetch','origin','main'],check=True)
subprocess.run(['git','-C',str(ROOT),'checkout','main'],check=True)
subprocess.run(git+['-C',str(ROOT),'pull','--ff-only','origin','main'],check=True)
subprocess.check_call([sys.executable,'-m','pip','install','-q','-r',str(ROOT/'requirements-colab.txt')])
print('main:',subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True).strip())


## 1. Prepare rich student metadata/features

This **does not rerun openSMILE** and does not recreate raw audio. It reads the existing USSD ComParE16 cache and creates only TRAIN-only normalization + compact text TF-IDF artifacts.


In [ ]:
DATA=Path('/content/drive/MyDrive/DAIC_WOZ')
COMPARE16=DATA/'experiments'/'ussd_compare16'
FEAT=DATA/'processed'/'participant_student'/'v2_rich'
OUT=DATA/'experiments'/'students'/'participant_v2_rich'/'no_kd'/'seed_103'

assert (COMPARE16/'participant_manifest.csv').exists(),COMPARE16/'participant_manifest.csv'

%cd /content/reliability-aware-depression-kd

!python -u -m scripts.students.participant_student_v2.prepare \
  --daic-root "{DATA}" \
  --compare16-root "{COMPARE16}" \
  --output "{FEAT}"


## 2. Verify the preparation audit


In [ ]:
summary=json.loads((FEAT/'feature_summary.json').read_text())
print(json.dumps(summary,indent=2))
assert summary['train_participants']==107
assert summary['dev_participants']==34
assert summary['participant_440_excluded'] is True
assert summary['text_features']==250
assert summary['test_opened'] is False and summary['test_prepared'] is False


## 3. Stage cached ComParE16 files to Colab local disk

This is optional but recommended. It copies existing Drive `.npy` files once so every epoch does not repeatedly read Google Drive. Nothing is stored in RAM and nothing is recomputed.


In [ ]:
from tqdm.auto import tqdm

LOCAL=Path('/content/rich_student_compare16')
for split in ['train','dev']:
    src=COMPARE16/'participant_features'/split
    dst=LOCAL/split; dst.mkdir(parents=True,exist_ok=True)
    files=sorted(src.glob('*.npy'))
    for f in tqdm(files,desc=f'Copy {split} ComParE16 to local disk',colour='green'):
        q=dst/f.name
        if not q.exists() or q.stat().st_size!=f.stat().st_size: shutil.copy2(f,q)
print('Local acoustic cache:',LOCAL)


## 4. Train the rich No-KD baseline

The threshold is fixed at 0.5. Checkpoint selection uses DEV-34 participant macro-F1 only.


In [ ]:
!python -u -m scripts.students.participant_student_v2.train_baseline \
  --features "{FEAT}" \
  --local-audio-root "{LOCAL}" \
  --output "{OUT}" \
  --seed 103 \
  --batch-size 8 \
  --epochs 100 \
  --patience 20 \
  --lr 2e-4 \
  --weight-decay 5e-4


## 5. Review DEV-34 and efficiency


In [ ]:
import pandas as pd
metrics=json.loads((OUT/'metrics.json').read_text())
complexity=json.loads((OUT/'model_complexity.json').read_text())
design=json.loads((OUT/'design_audit.json').read_text())
print('DEV-34'); print(json.dumps(metrics['dev34'],indent=2))
print('\nCOMPLEXITY'); print(json.dumps(complexity,indent=2))
print('\nDESIGN AUDIT'); print(json.dumps(design,indent=2))
display(pd.read_csv(OUT/'dev_predictions.csv'))
assert metrics['protocol']['teacher_predictions_loaded'] is False
assert metrics['protocol']['test_opened'] is False


## 6. Stop here

Freeze and inspect this No-KD result before rebuilding Standard KD and Reliability-Aware KD for v2. Do not use the old v0 KD notebook with this checkpoint.
